# Becke Partition 一阶梯度代码迁移准备

In [1]:
from pyscf import gto, dft, lib, grad, hessian, data
import numpy as np
from functools import partial
import scipy

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
def get_grids(xyz):
    mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()
    grids = dft.grid.Grids(mol)
    grids.radii_adjust = dft.radi.becke_atomic_radii_adjust
    grids.build(sort_grids=False)
    return mol, grids

In [3]:
xyz_orig = np.array([[0.0, 0.0, 0.0], [1.0, 0.1, 0.2], [0.3, 1.1, 0.2], [0.1, 0.1, 1.2]])  # in angstrom

def perturb_xyz(xyz, A, t, sgn, delta):
    xyz_pert = xyz.copy()
    xyz_pert[A, t] += sgn * delta
    return xyz_pert

def perturb_mol(A, t, sgn, delta):
    xyz_coords = perturb_xyz(xyz_orig, A, t, sgn, delta)
    atm_symbols = ["N", "H", "H", "H"]
    xyz_str = "\n".join(f"{atm} {x:.6f} {y:.6f} {z:.6f}" for atm, (x, y, z) in zip(atm_symbols, xyz_coords))
    mol_pert = gto.Mole(atom=xyz_str, basis="def2-TZVP", max_memory=32000).build()
    return mol_pert

def perturb_mol_grids(A, t, sgn, delta):
    mol_pert = perturb_mol(A, t, sgn, delta)
    grids_pert = dft.grid.Grids(mol_pert)
    grids_pert.radii_adjust = dft.radi.becke_atomic_radii_adjust
    grids_pert.build(sort_grids=False)
    return mol_pert, grids_pert

## 1. PySCF 解析格点一阶梯度 npz 文件保存

这份文档的目的是，要将 Becke Partition 一阶梯度的代码迁移到 Rust 中。为此，我们需要先在 Python 中准备好 Becke Partition 一阶梯度的 npz 文件，这样我们就可以在 Rust 中读取这些数据并进行验证。

In [4]:
mol, grids = perturb_mol_grids(A=0, t=0, sgn=1, delta=0.0)

In [5]:
nonpad_mask = grids.atm_idx != -1
quadrature_weights = grids.quadrature_weights[nonpad_mask]
grid_coords = grids.coords[nonpad_mask]
atm_coords = mol.atom_coords()
atm_indices = grids.atm_idx[nonpad_mask]
weights = grids.weights[nonpad_mask]

natm = atm_coords.shape[0]
ngrids = grid_coords.shape[0]
becke_radii_adjust = dft.radi.becke_atomic_radii_adjust(mol, grids.atomic_radii)
radii_table = np.array([becke_radii_adjust(i, j, 0) for i in range(natm) for j in range(natm)]).reshape(natm, natm)
dw_ref = hessian.rks.get_dweight_dA(mol, grids)[..., nonpad_mask]

In [6]:
becke_deriv1_dict = {
    "grids": grid_coords,
    "weights": weights,
    "atm_coords": atm_coords,
    "atm_indices": atm_indices.astype(float),
    "wquad": quadrature_weights,
    "radii_table": radii_table,
    "dw_ref": dw_ref,
}

np.savez("becke_deriv1_dict.npz", **becke_deriv1_dict)

## 2. 格点计算编写规则

该文档将对应 `10-2-becke_partition_deriv1.ipynb` 的实现；但为将代码迁移到 Rust，我们需要预先对 Python 代码的实现策略作重构。

Python 代码的策略是，为公式与程序对应，尽可能使用完整张量而非循环迭代计算。由于 NumPy 具有一定的向量化优化，这么做在纯 Python 下效率反而经常是更高的。

但 Rust 或其他编译语言下，这种策略会显示一些缺点，或者编译语言有其自己的特色：

1. 张量的内存占用大，至少是 memory footprint 大；用循环可以避免对较大张量的写入。
2. 基于 LLVM/GNU 的编译器，可以对特定 align 的向量作优化；向量化仍然是需要的，但我们需要的是 SIMD 循环，而非将整个问题放到一个张量里面。
3. 一些张量下不适合表示的、或不适合计算的问题 (譬如对称性的利用、高级索引等)，我们要在 Rust 的循环下处理。

其中的第 1/3 点是 Python 下可以 prototype 的；但众所周知 Python 下循环性能差，两种计算机语言将在两种代码风格下表现出不同的性能。第 2 点将在合理的循环设计下，在 Rust 自身内部去解决。

除此之外，我们在程序设计时需要遵循一些原则：

- 不能出现 $O(N^3)$ 内存用量。这里留意，尽管格点数量是 batch 的，与体系大小无关；但这个 batch 大小完全可能是数千。在合理的循环下，我们需要尽量避免 $n_\mathrm{atom}^2 n_\mathrm{batch}$ 大小的张量。

## 3. 零阶重新实现

我们可能将实现与 PySCF 现行的 C 程序不同的程序。

### 3.1 格点无关量

在整个计算中，我们可能对格点作下述划分：

- batch 级别：控制总内存用量，可以是 1k - 40k 级别。
- chunk 级别：并行单元，可以是每线程 256 - 4k 级别。
- SIMD 级别：单个向量化单元，通常是 8 (8 FP64 = 512 bits)。

不论我们采用 2 重还是 3 重划分，它们都是对格点的简单划分。格点之外的项呢？它们不能在格点循环中被重复计算。这些格点循环无关量本身也不占太多内存，我们需要在传入 becke partition 前就解决这些量。

- `wquad` $w_g^\text{quad}$：原始 Lebedev 权重，维度 $(g,)$ `(ngrids,)`

- `a` $a_{AB}$：Becke radii 矫正表，维度 $(A, B)$ `(natm, natm)`

In [7]:
wquad = quadrature_weights
a = radii_table

- `atm_coords` $R_{A t}$：原子坐标，维度 $(A, 3)$ `(natm, 3)`

- `atom_dist` $\Vert R \Vert_{AB}$：原子间距离，维度 $(A, B)$ `(natm, natm)`；其只作为分母出现，对角元设为 $\infty$，避免除零。

    $$
    \Vert R \Vert_{AB} = 
    \begin{cases}
    \sqrt{\sum_t (R_{B t} - R_{A t})^2} & A \neq B \\
    \infty & A = B
    \end{cases}
    $$

In [8]:
# atom_dist = np.linalg.norm(atm_coords[:, None, :] - atm_coords[None, :, :], axis=-1)  # rt::sci::cdist in RSTSR
atom_dist = scipy.spatial.distance.cdist(atm_coords, atm_coords)
for i in range(natm):
    atom_dist[i, i] = np.inf

### 3.2 switch 函数

最关键的辅助函数是 switch 函数 $s_{AB}$。

自变量为 $\nu$ 的函数：

$$
\begin{align}
p(\nu) &= \frac{3}{2} \nu - \frac{1}{2} \nu^3 \tag{19} \\
f_3(\nu) &= p \circ p \circ p (\nu) \tag{20} \\
s_3(\nu) &= \frac{1}{2} (1 - f_3(\nu)) \tag{21}
\end{align}
$$

自变量为 $\mu$ 的函数：

$$
\begin{align}
s (\mu) &= s_3 \circ \nu (\mu) \tag{A1} \\
\nu(\mu) &= \mu + a ( 1 - \mu^2 ) \tag{A2}
\end{align}
$$

一般来说，Becke partition 的函数 $p(\nu)$ 会作三次。我们可以对该函数作特殊特化；但对于 3 次的情况需要手动 unroll (或使用泛型，但我感觉不建议)。

请留意，我们计算的是 $f_3 (\nu)$，而不是最终的 $s(\mu)$。这是因为我们可以利用一次 $\mu_{M N t} = - \mu_{N M t}$ 的反对称特性；这个反对称特性维持到 $f_3 (\nu)$，但在 $s(\mu)$ 中被破坏了。

In [9]:
def switch_f3(mu, a_factor):
    nu = mu + a_factor * (1 - mu * mu)  # eq (A2)
    f1 = (1.5 - 0.5 * nu * nu) * nu     # eq (19)
    f2 = (1.5 - 0.5 * f1 * f1) * f1     # eq (19)
    f3 = (1.5 - 0.5 * f2 * f2) * f2     # eq (19)
    return f3

In [10]:
def switch_fhardness(mu, a_factor, hardness):
    nu = mu + a_factor * (1 - mu * mu)  # eq (A2)
    f = nu
    for _ in range(hardness):
        f = (1.5 - 0.5 * f * f) * f  # eq (19)
    return f

assert np.allclose(switch_f3(0.5, 0.1), switch_fhardness(0.5, 0.1, 3))

### 3.3 完整实现

In [11]:
w = np.zeros(ngrids)

nbatch_grid = 32
for g0 in range(0, ngrids, nbatch_grid):
    # handle batch
    g1 = min(g0 + nbatch_grid, ngrids)
    nbatch = g1 - g0
    batch_coords = grid_coords[g0:g1]
    batch_dist = scipy.spatial.distance.cdist(atm_coords, batch_coords)
    batch_atm_indices = atm_indices[g0:g1]

    # evaluate Becke partition function (production of switch functions)
    P = np.ones((natm, nbatch))
    for A in range(natm):
        for B in range(A):
            a_factor = a[A, B]
            mu = (batch_dist[A] - batch_dist[B]) / atom_dist[A, B]
            f3 = switch_f3(mu, a_factor)
            P[A] *= 0.5 * (1 - f3)
            P[B] *= 0.5 * (1 + f3)
    
    # compute partition function and weights
    Pg = np.zeros(nbatch)
    for g in range(nbatch):
        Pg[g] = P[batch_atm_indices[g], g]
    Z = P.sum(axis=0)
    partition = Pg / Z
    w[g0:g1] = partition * wquad[g0:g1]

assert np.allclose(w, weights)

## 4. 一阶重新实现

### 4.1 格点无关量

In [12]:
dR_atom_dist = (atm_coords[:, None, :] - atm_coords[None, :, :]) / atom_dist[:, :, None]
assert np.allclose(- dR_atom_dist.swapaxes(0, 1), dR_atom_dist)

### 4.2 switch 函数一阶导数

In [13]:
def switch_dnu_f3(mu, a_factor):
    nu = mu + a_factor * (1 - mu * mu)  # eq (A2)
    f1 = (1.5 - 0.5 * nu * nu) * nu     # eq (19)
    f2 = (1.5 - 0.5 * f1 * f1) * f1     # eq (19)
    f3 = (1.5 - 0.5 * f2 * f2) * f2     # eq (19)

    df1 = 1.5 * (1 - nu * nu)
    df2 = 1.5 * (1 - f1 * f1) * df1
    df3 = 1.5 * (1 - f2 * f2) * df2
    return f3, df3

In [ ]:
def switch_dnu_fhardness(mu, a_factor, hardness):
    nu = mu + a_factor * (1 - mu * mu)  # eq (A2)
    f = nu
    df = 1.0
    for _ in range(hardness):
        df = 1.5 * (1 - f * f) * df
        f = (1.5 - 0.5 * f * f) * f  # eq (19)
    return f, df

assert np.allclose(switch_dnu_f3(0.5, 0.1)[0], switch_dnu_fhardness(0.5, 0.1, 3)[0])
assert np.allclose(switch_dnu_f3(0.5, 0.1)[1], switch_dnu_fhardness(0.5, 0.1, 3)[1])

### 4.3 完整实现

In [ ]:
w = np.zeros(ngrids)
dw = np.zeros((natm, 3, ngrids))
INVTOL = 1e-14

nbatch_grid = 32
for g0 in range(0, ngrids, nbatch_grid):
    # handle batch
    g1 = min(g0 + nbatch_grid, ngrids)
    nbatch = g1 - g0
    # batch for 0-th order
    batch_coords = grid_coords[g0:g1]
    batch_dist = scipy.spatial.distance.cdist(atm_coords, batch_coords)
    batch_atm_indices = atm_indices[g0:g1]
    # transpose (for 1-st order grid-distance derivative)
    batch_coords = batch_coords.swapaxes(-1, -2)
    # batch for 1-st order
    dR_batch_dist = (atm_coords[:, :, None] - batch_coords[None, :, :]) / batch_dist[:, None]

    # Pass 1: evaluate Becke partition function P_{Mg} = prod_{N!=M} s_{MNg}.
    # r_g is treated as R-independent here; the total-derivative fix is applied to dw below.
    P = np.ones((natm, nbatch))
    dR_Z = np.zeros((natm, 3, nbatch))
    dR_Pg = np.zeros((natm, 3, nbatch))
    for A in range(natm):
        for B in range(A):
            a_factor = a[A, B]
            mu = (batch_dist[A] - batch_dist[B]) / atom_dist[A, B]
            f3 = switch_f3(mu, a_factor)
            P[A] *= 0.5 * (1 - f3)
            P[B] *= 0.5 * (1 + f3)

    # Pass 2: first-order derivatives dR_Z_{Atg}, dR_Pg_{Atg}.
    # Both contract the per-pair log-derivative dmu_log_s_{MN} * dR_mu_{MN} against the
    # *full* P_{Mg} (built in pass 1), so they must follow pass 1 rather than be folded
    # into the incremental product above.
    for A in range(natm):
        for B in range(A):
            a_factor = a[A, B]
            mu = (batch_dist[A] - batch_dist[B]) / atom_dist[A, B]
            # dR_mu_{ABg}/dR_{At} (role A) and dR_mu_{ABg}/dR_{Bt} (role B)
            dR_mu_roleA = (  dR_batch_dist[A] - mu * dR_atom_dist[A, B, :, None]) / atom_dist[A, B, None]
            dR_mu_roleB = (- dR_batch_dist[B] + mu * dR_atom_dist[A, B, :, None]) / atom_dist[A, B, None]

            f3, dnu_f3 = switch_dnu_f3(mu, a_factor)
            sA = 0.5 * (1 - f3)   # s_{ABg} = s(mu_{ABg})
            sB = 0.5 * (1 + f3)   # s_{BAg} = s(-mu_{ABg}); uses f3 antisymmetry, needs a_{BA} = -a_{AB}
            dmu_nu = 1 - 2 * a_factor * mu
            dmu_sA = -0.5 * dnu_f3 * dmu_nu   # dsA/dmu_{ABg}
            dmu_sB =  0.5 * dnu_f3 * dmu_nu   # dsB/dmu_{ABg}
            sA_safe = np.where(sA > INVTOL, sA, INVTOL)
            sB_safe = np.where(sB > INVTOL, sB, INVTOL)
            dmu_log_sA = dmu_sA / sA_safe
            dmu_log_sB = dmu_sB / sB_safe

            # dR_Z_{At} = sum_M P_{Mg} sum_N dmu_log_s_{MN} dR_mu_{MN}/dR_{At}.
            # For pair (A,B) both (M,N) orderings survive the sum; using mu_{BA} = -mu_{AB}
            # and a_{BA} = -a_{AB}, the log-derivative signs cancel against the opposite
            # dR_mu roles, leaving a single common factor times each role derivative:
            #   dR_Z[A] += (P_A dmu_log_sA + P_B dmu_log_sB) * dR_mu_roleA
            #   dR_Z[B] += (P_A dmu_log_sA + P_B dmu_log_sB) * dR_mu_roleB
            common_Z = P[A] * dmu_log_sA + P[B] * dmu_log_sB
            dR_Z[A] += common_Z[None, :] * dR_mu_roleA
            dR_Z[B] += common_Z[None, :] * dR_mu_roleB

            # dR_Pg_{At} = P_{A_g g} sum_N dmu_log_s_{A_g N} dR_mu_{A_g N}/dR_{At};
            # only grids owned by A or B pick up a contribution from this pair.
            maskA = batch_atm_indices == A
            maskB = batch_atm_indices == B
            common_Pg = (np.where(maskA, P[A] * dmu_log_sA, 0.0)
                       + np.where(maskB, P[B] * dmu_log_sB, 0.0))
            dR_Pg[A] += common_Pg[None, :] * dR_mu_roleA
            dR_Pg[B] += common_Pg[None, :] * dR_mu_roleB
            
    # compute partition function and weights
    Pg = np.zeros(nbatch)
    for g in range(nbatch):
        Pg[g] = P[batch_atm_indices[g], g]
    Z = P.sum(axis=0)
    partition = Pg / Z
    w[g0:g1] = partition * wquad[g0:g1]

    # grid-weight derivative dw (r_g fixed): quotient rule on P^g_{A_g} / Z_g
    dw_batch = wquad[g0:g1][None, None, :] * (dR_Pg / Z[None, None, :]
        - Pg[None, None, :] / Z[None, None, :]**2 * dR_Z)
    # total-derivative fix for g in A_g: dw_{A_g} = -sum_{C != A_g} dw_C
    for g in range(nbatch):
        Ag = batch_atm_indices[g]
        sum_excl = dw_batch[:, :, g].sum(axis=0) - dw_batch[Ag, :, g]
        dw_batch[Ag, :, g] = -sum_excl
    dw[:, :, g0:g1] = dw_batch

assert np.allclose(w, weights)
assert np.allclose(dw, dw_ref)